# Pattern Learning Data Preparation Pipeline Walkthrough

This notebook runs the **near-miss / hazard pattern learning** data preparation pipeline interactively.

It uses the same functions from the existing project scripts and displays raw dataset samples, intermediate joined dataframes, final ML-ready output dataframes, EDA summary tables, and EDA plots.

Default display size is **100 rows per dataframe**.

## 1. Configure paths

Update `INPUT_DIR` if your raw CSV files are stored somewhere else. The notebook expects the six exported CSV files: `INCIDENT_VIEW.csv`, `INCIDENTINJURY_VIEW.csv`, `LISTITEM_VIEW.csv`, `LOCATION_VIEW.csv`, `TASK_VIEW.csv`, and `AUDIT_VIEW.csv`.

In [23]:
from pathlib import Path
import sys
import gc
import pandas as pd
from IPython.display import display, Markdown, Image

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
INPUT_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR = OUTPUT_DIR / "processed"
EDA_DIR = OUTPUT_DIR / "eda"

REFERENCE_DATE = pd.Timestamp("2026-05-20", tz="UTC")
ACTIVE_ONLY = True
DISPLAY_ROWS = 100

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EDA_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Project root: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ehs-tan-dev/code/Users/tan.cheng/EHS_predictive_modeling/pattern_learning_project
Input directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ehs-tan-dev/code/Users/tan.cheng/EHS_predictive_modeling/pattern_learning_project/data/raw
Output directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ehs-tan-dev/code/Users/tan.cheng/EHS_predictive_modeling/pattern_learning_project/outputs


## 2. Import pipeline functions

The notebook keeps the original code modular. It imports the functions already defined in the project scripts instead of duplicating logic inside the notebook.

In [1]:
from utils import read_csv_safely, write_dataframe
from lookups import prepare_listitems
from locations import build_location_hierarchy
from features import (
    prepare_injury_agg,
    prepare_incidents,
    prepare_pattern_learning_records,
    prepare_incident_injury_all_records,
    prepare_tasks,
    prepare_audits,
    make_site_department_month_features,
)
from eda import create_eda_outputs
from run_data_prep import (
    RAW_FILE_NAMES,
    compact_incident_for_eda,
    compact_task_for_eda,
    compact_audit_for_eda,
)

## 3. Helper display functions

In [14]:
def display_df(name: str, df: pd.DataFrame, n: int = DISPLAY_ROWS):
    display(Markdown(f"### {name}"))
    display(Markdown(f"**Shape:** `{df.shape[0]:,}` rows × `{df.shape[1]:,}` columns"))
    display(df.head(n))


def read_required(input_dir: Path, key: str) -> pd.DataFrame:
    file_name = RAW_FILE_NAMES[key]
    path = input_dir / file_name
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    df = read_csv_safely(path)
    print(f"Loaded {file_name}: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    return df


def show_csv_table(path: Path, n: int = DISPLAY_ROWS):
    if path.exists():
        df = pd.read_csv(path)
        display_df(path.name, df, n=n)
    else:
        print(f"Missing table: {path}")


def show_plot(path: Path):
    if path.exists():
        display(Markdown(f"### {path.name}"))
        display(Image(filename=str(path)))
    else:
        print(f"Missing plot: {path}")

## 4. Load raw datasets and show samples

These are the original exported files before joins or feature engineering.

In [15]:
raw_shapes = {}

listitem_raw = read_required(INPUT_DIR, "listitem")
location_raw = read_required(INPUT_DIR, "location")
injury_raw = read_required(INPUT_DIR, "injury")
incident_raw = read_required(INPUT_DIR, "incident")
task_raw = read_required(INPUT_DIR, "task")
audit_raw = read_required(INPUT_DIR, "audit")

raw_dfs = {
    "LISTITEM_VIEW raw": listitem_raw,
    "LOCATION_VIEW raw": location_raw,
    "INCIDENTINJURY_VIEW raw": injury_raw,
    "INCIDENT_VIEW raw": incident_raw,
    "TASK_VIEW raw": task_raw,
    "AUDIT_VIEW raw": audit_raw,
}

raw_shapes = {
    "listitem": listitem_raw.shape,
    "location": location_raw.shape,
    "injury": injury_raw.shape,
    "incident": incident_raw.shape,
    "task": task_raw.shape,
    "audit": audit_raw.shape,
}

for name, df in raw_dfs.items():
    display_df(name, df, n=DISPLAY_ROWS)

Loaded LISTITEM_VIEW.csv: 4,074 rows × 12 columns
Loaded LOCATION_VIEW.csv: 1,341 rows × 18 columns
Loaded INCIDENTINJURY_VIEW.csv: 7,746 rows × 44 columns
Loaded INCIDENT_VIEW.csv: 167,061 rows × 68 columns
Loaded TASK_VIEW.csv: 161,425 rows × 39 columns
Loaded AUDIT_VIEW.csv: 490,365 rows × 33 columns


### LISTITEM_VIEW raw

**Shape:** `4,074` rows × `12` columns

,LISTITEMID,LISTTYPECODE,PARENTID,CLIENTID,LOCATIONID,LISTORDER,CODE,ITEM,SHORTNAME,DESCRIPTION,ACTIVE,HISTORYID
0,-1,dummy,NaN,NaN,NaN,-1.0,DMY,DUMMY VALUE,NaN,NaN,True,19095.0
1,3,role,NaN,NaN,NaN,NaN,SYSTEM-VIEW,EHS Only View Access,View,Persons assigned privileges to view data,True,19096.0
2,4,role,NaN,NaN,NaN,NaN,LOCATION-ADMINISTRATOR-SYSTEM,System Administrator,Administrator,System Administrator for a location,True,19097.0
3,5,locationcategory,NaN,NaN,NaN,NaN,ROOT,Root,Root,The category of a node representing the root o...,True,19098.0
4,6,locationcategory,NaN,NaN,NaN,NaN,REPORTING,Reporting,Reporting,A node of the EHS reporting location tree,True,19099.0
...,...,...,...,...,...,...,...,...,...,...,...,...
95,474,incidenttype,NaN,NaN,NaN,0.0,ENVIRONMENTAL,Environmental Release,Environmental,A spill or release of a substance,True,19190.0
96,475,incidenttype,NaN,NaN,NaN,0.0,FIRE,Fire,Fire,A destructive or uncontrolled burning of any m...,True,19191.0
97,476,incidenttype,NaN,NaN,NaN,0.0,EXPLOSION,Explosion,Explosion,A sudden or violent release of mechanical or c...,True,19192.0
98,477,incidenttype,NaN,NaN,NaN,0.0,EQUIPMENT,Equipment Failure,Equipment Failure,A failure of equipment,True,19193.0


### LOCATION_VIEW raw

...

**Shape:** `161,425` rows × `39` columns

,VERIFICATIONDUEDATE,STARTDATE,SOURCEDATE,REVISEDDUEDATE,MARKEDCOMPLETEDATE,COMPLETIONDATE,ASSIGNEDDATE,DUEDATE,TASKID,PARENTTASKID,...,EXTERNALNUMBER,OTHERLOCATIONNAME,EQUIPMENT,DESCRIPTION,BESTPRACTICES,PERMITSECTION,VERIFICATIONREASON,ACTIVE,ARCHIVED,HISTORYID
0,NaN,NaN,NaN,NaN,NaN,2016-08-10T00:00:00Z,2016-08-10T00:00:00Z,2016-08-10T00:00:00Z,1,NaN,...,NaN,NaN,NaN,This record has been entered during the transi...,NaN,NaN,NaN,False,True,431
1,NaN,NaN,NaN,NaN,NaN,2016-08-10T00:00:00Z,2016-08-10T00:00:00Z,2016-08-10T00:00:00Z,2,NaN,...,NaN,NaN,NaN,This record has been entered between migration...,NaN,NaN,NaN,False,True,433
2,NaN,NaN,NaN,NaN,NaN,2016-08-10T00:00:00Z,2016-08-10T00:00:00Z,2016-08-10T00:00:00Z,3,NaN,...,NaN,NaN,NaN,This entry has been created following the migr...,NaN,NaN,NaN,False,True,435
3,NaN,NaN,NaN,NaN,NaN,NaN,2016-08-15T00:13:33.837Z,2016-09-30T00:00:00Z,4,NaN,...,NaN,NaN,NaN,Please review this record entry as it has been...,NaN,NaN,NaN,False,True,437
4,NaN,NaN,NaN,NaN,NaN,2016-08-23T00:00:00Z,2016-08-22T00:00:00Z,2016-08-26T00:00:00Z,5,NaN,...,NaN,NaN,NaN,Arrange for hose to be mounted via hose reel.,NaN,NaN,NaN,False,True,439
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,NaN,NaN,NaN,NaN,NaN,2016-08-05T00:00:00Z,2016-10-26T00:00:00Z,2016-10-26T00:00:00Z,96,NaN,...,NaN,NaN,NaN,Discussed eyes on take and awareness of surrou...,NaN,NaN,NaN,True,False,3638
96,NaN,NaN,NaN,NaN,NaN,2016-10-11T00:00:00Z,2016-10-26T00:00:00Z,2016-11-04T00:00:00Z,97,NaN,...,NaN,NaN,NaN,Please review the information entered for this...,NaN,NaN,NaN,True,False,3639
97,NaN,NaN,NaN,NaN,NaN,2016-10-31T00:00:00Z,2016-10-26T00:00:00Z,2016-11-04T00:00:00Z,98,NaN,...,NaN,NaN,NaN,Please review the information entered for this...,NaN,NaN,NaN,True,False,3642
98,NaN,NaN,NaN,NaN,NaN,2016-11-16T00:00:00Z,2016-10-26T00:00:00Z,2016-11-04T00:00:00Z,99,NaN,...,NaN,NaN,NaN,Please review all information entered for this...,NaN,NaN,NaN,True,False,3646


### AUDIT_VIEW raw

**Shape:** `490,365` rows × `33` columns

,ACTUALEND,ACTUALSTART,SCHEDULEDEND,SCHEDULEDSTART,AUDITID,PARENTID,ROOTAUDITID,CLIENTID,AUDITCATEGORYID,AUDITTYPEID,...,SHORTNAME,OTHERLOCATIONNAME,EXTERNALID,TITLE,DESCRIPTION,ASSOCIATEDPARTIES,COMMENTS,ACTIVE,ARCHIVED,HISTORYID
0,NaN,NaN,NaN,NaN,1,NaN,1,FEB8B939-BDC0-DD11-8038-DFD6234E2644,787,NaN,...,NaN,NaN,NaN,EHS Process Audit,NaN,NaN,NaN,True,False,380
1,NaN,NaN,NaN,NaN,2,NaN,2,FEB8B939-BDC0-DD11-8038-DFD6234E2644,787,NaN,...,NaN,NaN,NaN,General Fire Prevention Checklist,NaN,NaN,NaN,True,False,381
2,NaN,NaN,NaN,NaN,3,NaN,3,FEB8B939-BDC0-DD11-8038-DFD6234E2644,787,NaN,...,NaN,NaN,NaN,EHS Inspection Form MSHA,NaN,NaN,NaN,True,False,382
3,NaN,NaN,NaN,NaN,4,NaN,4,FEB8B939-BDC0-DD11-8038-DFD6234E2644,787,NaN,...,NaN,NaN,NaN,EHS Observations U.S. Warehouse Operations Che...,NaN,NaN,NaN,True,False,383
4,NaN,NaN,NaN,NaN,5,NaN,5,FEB8B939-BDC0-DD11-8038-DFD6234E2644,787,NaN,...,NaN,NaN,NaN,EHS Inspection Form OSHA,NaN,NaN,NaN,True,False,384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,NaN,NaN,NaN,2016-11-02T19:32:53.267Z,96,25.0,25,FEB8B939-BDC0-DD11-8038-DFD6234E2644,697,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,4266
96,NaN,NaN,NaN,2016-11-02T20:19:40.550Z,97,25.0,25,FEB8B939-BDC0-DD11-8038-DFD6234E2644,697,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,4279
97,NaN,NaN,NaN,2016-11-02T20:28:51.387Z,98,25.0,25,FEB8B939-BDC0-DD11-8038-DFD6234E2644,697,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,4286
98,NaN,NaN,NaN,2016-11-02T08:00:00Z,99,25.0,25,FEB8B939-BDC0-DD11-8038-DFD6234E2644,697,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,4289


## 5. Prepare lookup table from LISTITEM_VIEW

Purpose: decode numeric IDs into stakeholder-readable names such as incident category, incident status, audit type, task status, etc.

In [ ]:
listitems = prepare_listitems(listitem_raw)
display_df("Prepared listitem lookup", listitems, n=DISPLAY_ROWS)

### Prepared listitem lookup

**Shape:** `4,074` rows × `7` columns

,list_item_id,list_type_code,code,item,shortname,description,active
0,-1,dummy,DMY,DUMMY VALUE,<NA>,NaN,True
1,3,role,SYSTEM-VIEW,EHS Only View Access,<NA>,Persons assigned privileges to view data,True
2,4,role,LOCATION-ADMINISTRATOR-SYSTEM,System Administrator,<NA>,System Administrator for a location,True
3,5,locationcategory,ROOT,Root,<NA>,The category of a node representing the root o...,True
4,6,locationcategory,REPORTING,Reporting,<NA>,A node of the EHS reporting location tree,True
...,...,...,...,...,...,...,...
95,474,incidenttype,ENVIRONMENTAL,Environmental Release,<NA>,A spill or release of a substance,True
96,475,incidenttype,FIRE,Fire,<NA>,A destructive or uncontrolled burning of any m...,True
97,476,incidenttype,EXPLOSION,Explosion,<NA>,A sudden or violent release of mechanical or c...,True
98,477,incidenttype,EQUIPMENT,Equipment Failure,<NA>,A failure of equipment,True


## 6. Build location hierarchy from LOCATION_VIEW

Purpose: attach each record to site, department, country, region, and business unit.

Key joins used later:

```text
INCIDENT_VIEW.LOCATIONID = LOCATION_VIEW.LOCATIONID
TASK_VIEW.LOCATIONID = LOCATION_VIEW.LOCATIONID
AUDIT_VIEW.SCHEDULEDLOCATIONID = LOCATION_VIEW.LOCATIONID
```

In [ ]:
location_hierarchy = build_location_hierarchy(location_raw, listitems)
display_df("Location hierarchy", location_hierarchy, n=DISPLAY_ROWS)

### Location hierarchy

**Shape:** `1,341` rows × `29` columns

,location_id,location_name,location_code,parent_location_id,location_type_id,location_type_name,location_category_id,location_category_name,location_status_id,location_status_name,...,department_name,department_2_name,reporting_location_level_9_name,location_path,location_id_path,site_name_filled,department_name_filled,business_unit_name_filled,country_name_filled,region_name_filled
0,1,Root,NaN,NaN,8,Root,5,Root,NaN,None,...,<NA>,<NA>,<NA>,Root,1,Root,Root,Unknown,Unknown,Unknown
1,2,Default Client Root,NaN,1.0,7,Client,5,Root,NaN,None,...,<NA>,<NA>,<NA>,Root > Default Client Root,1 > 2,Default Client Root,Default Client Root,Unknown,Unknown,Unknown
2,3,Default Client Contracting Companies,NaN,2.0,8,Root,68,Contracting Company,NaN,None,...,<NA>,<NA>,<NA>,Root > Default Client Root > Default Client Co...,1 > 2 > 3,Default Client Contracting Companies,Default Client Contracting Companies,Unknown,Unknown,Unknown
3,4,Default Client Agencies,NaN,2.0,8,Root,125,Agency,NaN,None,...,<NA>,<NA>,<NA>,Root > Default Client Root > Default Client Ag...,1 > 2 > 4,Default Client Agencies,Default Client Agencies,Unknown,Unknown,Unknown
4,5,Default Client Reporting Locations,NaN,2.0,8,Root,6,Reporting,NaN,None,...,<NA>,<NA>,<NA>,Root > Default Client Root > Default Client Re...,1 > 2 > 5,Default Client Reporting Locations,Default Client Reporting Locations,Unknown,Unknown,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1491,Antamina,ANT,1490.0,503,Site,6,Reporting,NaN,None,...,<NA>,<NA>,<NA>,Root > Default Client Root > Default Client Re...,1 > 2 > 5 > 1419 > 1420 > 1432 > 1454 > 1490 >...,Antamina,Antamina,Surface,Peru,Latin America
96,1492,Antapaccay,ATP,1490.0,503,Site,6,Reporting,NaN,None,...,<NA>,<NA>,<NA>,Root > Default Client Root > Default Client Re...,1 > 2 > 5 > 1419 > 1420 > 1432 > 1454 > 1490 >...,Antapaccay,Antapaccay,Surface,Peru,Latin America
97,1493,Cerro Verde,CRV,1490.0,503,Site,6,Reporting,NaN,None,...,<NA>,<NA>,<NA>,Root > Default Client Root > Default Client Re...,1 > 2 > 5 > 1419 > 1420 > 1432 > 1454 > 1490 >...,Cerro Verde,Cerro Verde,Surface,Peru,Latin America
98,1494,Cuajone,CUA,1490.0,503,Site,6,Reporting,NaN,None,...,<NA>,<NA>,<NA>,Root > Default Client Root > Default Client Re...,1 > 2 > 5 > 1419 > 1420 > 1432 > 1454 > 1490 >...,Cuajone,Cuajone,Surface,Peru,Latin America


## 7. Aggregate injury records to one row per incident

Purpose: convert `INCIDENTINJURY_VIEW` from injury-level records to incident-level severity flags.

Key join used later:

```text
INCIDENT_VIEW.INCIDENTID = INCIDENTINJURY_VIEW.INCIDENTID
```

Engineered severity flag:

```text
severe_actual = fatality OR lost_time OR restricted_time OR inpatient
```

This is prepared for later severe-injury similarity modeling. It should **not** be used as an input feature in the first unsupervised clustering model.

In [ ]:
injury_agg = prepare_injury_agg(injury_raw)
display_df("Injury aggregation by incident", injury_agg, n=DISPLAY_ROWS)

### Injury aggregation by incident

**Shape:** `7,746` rows × `9` columns

,incident_id,injury_count,lost_time_any,restricted_time_any,fatality_any,emergency_room_any,inpatient_any,fatality_date,severe_actual
0,1,1,False,False,False,False,False,NaT,False
1,2,1,False,False,False,False,False,NaT,False
2,3,1,False,False,False,False,False,NaT,False
3,4,1,True,False,False,True,False,NaT,True
4,5,1,False,False,False,False,False,NaT,False
...,...,...,...,...,...,...,...,...,...
95,96,1,False,False,False,False,False,NaT,False
96,97,1,False,False,False,False,False,NaT,False
97,98,1,False,False,False,False,False,NaT,False
98,99,1,False,False,False,False,False,NaT,False


## 8. Prepare and enrich incidents

This step creates the first major joined dataframe.

Join operations:

1. Decode incident category/status using `LISTITEM_VIEW`
2. Join location hierarchy using `LOCATIONID`
3. Join injury aggregation using `INCIDENTID`

Main engineered fields: `ml_text_early`, `ml_text_full`, `text_early_word_count`, `incident_month`, `incident_week`, `is_active_record`, `is_pattern_candidate`, `has_usable_early_text`, and `severe_actual`.

In [21]:
incident_enriched = prepare_incidents(
    incident_raw,
    listitems=listitems,
    location_hierarchy=location_hierarchy,
    injury_agg=injury_agg,
    reference_date=REFERENCE_DATE,
)

display_df("Incident enriched - full intermediate dataframe", incident_enriched, n=DISPLAY_ROWS)

incident_compact = compact_incident_for_eda(incident_enriched)
display_df("Incident enriched - compact EDA view", incident_compact, n=DISPLAY_ROWS)

/mnt/batch/tasks/shared/LS_root/mounts/clusters/ehs-tan-dev/code/Users/tan.cheng/EHS_predictive_modeling/pattern_learning_project/src/features.py:140: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  inc[col] = inc[col].fillna(False).astype(bool)
/mnt/batch/tasks/shared/LS_root/mounts/clusters/ehs-tan-dev/code/Users/tan.cheng/EHS_predictive_modeling/pattern_learning_project/src/features.py:140: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  inc[col] = inc[col].fillna(False).astype(bool)
/mnt/batch/tasks/shared/LS_root/mounts/clusters/ehs-tan-dev/code/Users/t

### Incident enriched - full intermediate dataframe

**Shape:** `167,061` rows × `133` columns

,report_date,investigation_start_date,insurance_notified_date,incident_date,end_date,drug_test_date,alcohol_test_date,incident_id,client_id,incident_category_id,...,incident_month,incident_week,incident_date_missing,incident_date_after_reference,incident_date_before_2000,report_lag_days,is_active_record,is_pattern_candidate,has_location_match,has_usable_early_text
0,2016-08-10 00:00:01+00:00,NaT,NaT,2016-06-24 10:30:00+00:00,NaT,NaT,NaT,1,FEB8B939-BDC0-DD11-8038-DFD6234E2644,460,...,2016-06-01,2016-06-21,False,False,False,46.0,False,False,True,True
1,2016-08-10 00:00:01+00:00,NaT,NaT,2016-06-15 12:00:00+00:00,NaT,NaT,NaT,2,FEB8B939-BDC0-DD11-8038-DFD6234E2644,460,...,2016-06-01,2016-06-14,False,False,False,55.0,False,False,True,True
2,2016-08-10 00:00:01+00:00,NaT,NaT,2016-07-04 09:30:00+00:00,NaT,NaT,NaT,3,FEB8B939-BDC0-DD11-8038-DFD6234E2644,460,...,2016-07-01,2016-06-28,False,False,False,36.0,False,False,True,True
3,2016-08-14 00:00:01+00:00,NaT,NaT,2016-06-28 18:00:00+00:00,NaT,NaT,NaT,4,FEB8B939-BDC0-DD11-8038-DFD6234E2644,460,...,2016-06-01,2016-06-28,False,False,False,46.0,False,False,True,True
4,2016-08-22 00:00:01+00:00,NaT,NaT,2016-08-10 01:30:00+00:00,NaT,NaT,NaT,5,FEB8B939-BDC0-DD11-8038-DFD6234E2644,460,...,2016-08-01,2016-08-09,False,False,False,11.0,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,NaT,NaT,NaT,2014-09-15 10:57:00+00:00,NaT,NaT,NaT,96,FEB8B939-BDC0-DD11-8038-DFD6234E2644,460,...,2014-09-01,2014-09-09,False,False,False,NaN,True,False,True,True
96,NaT,NaT,NaT,2014-09-03 10:51:00+00:00,NaT,NaT,NaT,97,FEB8B939-BDC0-DD11-8038-DFD6234E2644,460,...,2014-09-01,2014-09-02,False,False,False,NaN,True,False,True,True
97,NaT,NaT,NaT,2014-09-03 10:46:00+00:00,NaT,NaT,NaT,98,FEB8B939-BDC0-DD11-8038-DFD6234E2644,460,...,2014-09-01,2014-09-02,False,False,False,NaN,True,False,True,True
98,NaT,NaT,NaT,2014-09-02 10:24:00+00:00,NaT,NaT,NaT,99,FEB8B939-BDC0-DD11-8038-DFD6234E2644,460,...,2014-09-01,2014-09-02,False,False,False,NaN,True,False,True,True


### Incident enriched - compact EDA view

**Shape:** `167,061` rows × `25` columns

,incident_id,incident_date,incident_month,incident_category_name,incident_status_name,location_id,site_name_filled,department_name_filled,business_unit_name_filled,country_name_filled,...,fatality_any,emergency_room_any,inpatient_any,severe_actual,text_early_word_count,has_location_match,incident_date_missing,incident_date_after_reference,incident_date_before_2000,report_lag_days
0,1,2016-06-24 10:30:00+00:00,2016-06-01,Incident,Closed,1661,Fielding,Fielding,Underground,Canada UG HR,...,False,False,False,False,166,True,False,False,False,46.0
1,2,2016-06-15 12:00:00+00:00,2016-06-01,Incident,Closed,1656,Sunderland,Sunderland,Underground,UK UG SR,...,False,False,False,False,56,True,False,False,False,55.0
2,3,2016-07-04 09:30:00+00:00,2016-07-01,Incident,Closed,1658,Worcester Operations,Worcester Operations,Underground,UK UG SR,...,False,False,False,False,45,True,False,False,False,36.0
3,4,2016-06-28 18:00:00+00:00,2016-06-01,Incident,Pending Closure,1645,Saint-Priest,Saint-Priest,Underground,France,...,False,True,False,True,56,True,False,False,False,46.0
4,5,2016-08-10 01:30:00+00:00,2016-08-01,Incident,Closed,1619,Moss Vale Operations,Moss Vale Operations,Underground,Australia,...,False,False,False,False,82,True,False,False,False,11.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,2014-09-15 10:57:00+00:00,2014-09-01,Incident,Closed,1702,Calgary Mine Air,Calgary Mine Air,KNA Service,Canada Surface,...,False,False,False,False,15,True,False,False,False,NaN
96,97,2014-09-03 10:51:00+00:00,2014-09-01,Incident,Closed,1702,Calgary Mine Air,Calgary Mine Air,KNA Service,Canada Surface,...,False,False,False,False,20,True,False,False,False,NaN
97,98,2014-09-03 10:46:00+00:00,2014-09-01,Incident,Closed,1702,Calgary Mine Air,Calgary Mine Air,KNA Service,Canada Surface,...,False,False,False,False,8,True,False,False,False,NaN
98,99,2014-09-02 10:24:00+00:00,2014-09-01,Incident,Closed,1702,Calgary Mine Air,Calgary Mine Air,KNA Service,Canada Surface,...,False,False,False,False,13,True,False,False,False,NaN


## 9. Prepare the first ML-ready table: near-miss and hazard pattern records

This filters `incident_enriched` to near misses and hazard identifications, active/non-archived records only, and records with usable early text. This is the main input table for the first pattern-learning model.

## 9. Prepare the first ML-ready table: near-miss and hazard pattern records

This filters `incident_enriched` to near misses and hazard identifications, active/non-archived records only, and records with usable early text. This is the main input table for the first pattern-learning model.

In [ ]:
pattern_records = prepare_pattern_learning_records(incident_enriched, active_only=ACTIVE_ONLY)
display_df("Pattern learning records - ML input table", pattern_records, n=DISPLAY_ROWS)
print(pattern_records.columns.tolist())

### Pattern learning records - ML input table

**Shape:** `141,958` rows × `55` columns

,incident_id,incident_number,incident_date,incident_month,incident_week,report_date,incident_category_id,incident_category_name,incident_status_id,incident_status_name,location_id,location_name,location_type_name,location_path,business_unit_name,region_name,country_name,site_name,department_name,department_2_name,site_name_filled,department_name_filled,business_unit_name_filled,country_name_filled,region_name_filled,title,description,equipment,vehicle,off_premises_location,other_process,other_activity,activity_during_incident,ml_text_early,ml_text_full,text_early_char_count,text_early_word_count,text_full_char_count,text_full_word_count,on_premises,work_related,process_safety,preventable,include_in_statistics,injury_count,lost_time_any,restricted_time_any,fatality_any,emergency_room_any,inpatient_any,severe_actual,is_active_record,incident_date_after_reference,incident_date_before_2000,report_lag_days
0,1839,NE-20130919-001,2013-09-19 23:00:00+00:00,2013-09-01,2013-09-17,2013-09-19 00:00:00+00:00,462,Near Miss,467,Closed,1606,"LeTourneau- Kalama, WA",Site,Root > Default Client Root > Default Client Re...,Manufacturing / Company Stores / Corporate,North America,USA,"LeTourneau- Kalama, WA",<NA>,<NA>,"LeTourneau- Kalama, WA","LeTourneau- Kalama, WA",Manufacturing / Company Stores / Corporate,USA,North America,Employee fell down,Employee was exiting an office and was placing...,,,Main Office,,,,Employee fell down Employee was exiting an off...,Employee fell down Employee was exiting an off...,220,44,220,44,True,<NA>,<NA>,<NA>,True,1,False,False,False,False,False,False,True,False,False,-1.0
1,1842,NE-20131123-001,2013-11-23 07:00:00+00:00,2013-11-01,2013-11-19,2013-11-26 00:00:00+00:00,462,Near Miss,467,Closed,1606,"LeTourneau- Kalama, WA",Site,Root > Default Client Root > Default Client Re...,Manufacturing / Company Stores / Corporate,North America,USA,"LeTourneau- Kalama, WA",<NA>,<NA>,"LeTourneau- Kalama, WA","LeTourneau- Kalama, WA",Manufacturing / Company Stores / Corporate,USA,North America,Employee hit by paint booth door.,Employee noticed the wind blew the paint booth...,,,Paint booth,,,Closing paint booth doors,Employee hit by paint booth door. Employee not...,Employee hit by paint booth door. Employee not...,297,52,385,64,True,<NA>,<NA>,<NA>,True,1,False,False,False,False,False,False,True,False,False,2.0
2,1859,NE-20121020-001,2012-10-20 15:45:00+00:00,2012-10-01,2012-10-16,2012-10-20 00:00:00+00:00,462,Near Miss,467,Closed,1572,875 - Longview Special Machining,Department 2,Root > Default Client Root > Default Client Re...,Surface,North America Surface,United States,Longview - Mining,Machining - Longview,875 - Longview Special Machining,Longview - Mining,Machining - Longview,Surface,United States,North America Surface,Plate clamp failure.,Jon Hollingsworth was using a plate clamp to l...,plate clamp,,RT 1600 East,,,,Plate clamp failure. Jon Hollingsworth was usi...,Plate clamp failure. Jon Hollingsworth was usi...,277,51,371,66,True,<NA>,<NA>,<NA>,True,0,False,False,False,False,False,False,True,False,False,-1.0
3,1862,NE-20121023-001,2012-10-23 17:20:00+00:00,2012-10-01,2012-10-23,2012-10-23 00:00:00+00:00,462,Near Miss,467,Closed,1562,865 - Longview Small Parts Fab,Department 2,Root > Default Client Root > Default Client Re...,Surface,North America Surface,United States,Longview - Mining,Fabrication - Longview,865 - Longview Small Parts Fab,Longview - Mining,Fabrication - Longview,Surface,United States,North America Surface,Dropped Part Due to Operator Error and Inadequ...,An employee was repositioning a large part on ...,Jib Crane and two Double chain and D-ring setups.,Jib Crane Asset#2410,Between areas 32D and 33D,,,,Dropped Part Due to Operator Error and Inadequ...,Dropped Part Due to Operator Error and Inadequ...,852,161,852,161,True,<NA>,<NA>,<NA>,True,0,False,False,False,False,False,False,True,False,False,-1.0
4,1867,NE-20121025-001,2012-10-25 22:55:00+00:00,2012-10-01,2012-10-23,2012-10-26 00:00:00+00:00,462,Nea

['incident_id', 'incident_number', 'incident_date', 'incident_month', 'incident_week', 'report_date', 'incident_category_id', 'incident_category_name', 'incident_status_id', 'incident_status_name', 'location_id', 'location_name', 'location_type_name', 'location_path', 'business_unit_name', 'region_name', 'country_name', 'site_name', 'department_name', 'department_2_name', 'site_name_filled', 'department_name_filled', 'business_unit_name_filled', 'country_name_filled', 'region_name_filled', 'title', 'description', 'equipment', 'vehicle', 'off_premises_location', 'other_process', 'other_activity', 'activity_during_incident', 'ml_text_early', 'ml_text_full', 'text_early_char_count', 'text_early_word_count', 'text_full_char_count', 'text_full_word_count', 'on_premises', 'work_related', 'process_safety', 'preventable', 'include_in_statistics', 'injury_count', 'lost_time_any', 'restricted_time_any', 'fatality_any', 'emergency_room_any', 'inpatient_any', 'severe_actual', 'is_active_record', '

## 10. Prepare all INCIDENT_VIEW rows enriched with injury information

    Return all incident rows after incident/injury/location/listitem enrichment.
    This is the intermediate analytical table before pattern-learning filters are applied.

In [ ]:
incident_injury_all_records = prepare_incident_injury_all_records(incident_enriched, active_only=ACTIVE_ONLY)
display_df("Incident-injury records - All", incident_injury_all_records, n=DISPLAY_ROWS)
print(incident_injury_all_records.columns.tolist())

NameError: name 'incident_enriched' is not defined

## 11. Prepare tasks / corrective actions

Join operations:

1. Decode task category/status/source type using `LISTITEM_VIEW`
2. Join location hierarchy using `TASK_VIEW.LOCATIONID = LOCATION_VIEW.LOCATIONID`

Main engineered fields: `task_text`, `is_open`, `is_closed`, `is_overdue`, `days_open`, `days_overdue`, `days_until_due`, and `task_event_month`.

In [ ]:
task_enriched = prepare_tasks(
    task_raw,
    listitems=listitems,
    location_hierarchy=location_hierarchy,
    reference_date=REFERENCE_DATE,
)

display_df("Task enriched - full intermediate dataframe", task_enriched, n=DISPLAY_ROWS)

task_compact = compact_task_for_eda(task_enriched)
display_df("Task enriched - compact EDA view", task_compact, n=DISPLAY_ROWS)

## 12. Prepare audits / inspections / observations

Join operations:

1. Decode audit category/type/status using `LISTITEM_VIEW`
2. Join location hierarchy using `AUDIT_VIEW.SCHEDULEDLOCATIONID = LOCATION_VIEW.LOCATIONID`

Main engineered fields: `audit_text`, `audit_event_month`, `is_observation`, `is_inspection`, `is_risk_assessment`, `is_unsafe_act`, `is_unsafe_condition`, `is_safe_act`, and `is_safe_condition`.

In [ ]:
audit_enriched = prepare_audits(
    audit_raw,
    listitems=listitems,
    location_hierarchy=location_hierarchy,
)

display_df("Audit enriched - full intermediate dataframe", audit_enriched, n=DISPLAY_ROWS)

audit_compact = compact_audit_for_eda(audit_enriched)
display_df("Audit enriched - compact EDA view", audit_compact, n=DISPLAY_ROWS)

## 13. Create site / department / month joined feature table

This table joins monthly aggregations from incidents, tasks, and audits using the common analytical keys: site, department, business unit, country, region, and month. This table is useful for EDA now and future supervised site-risk forecasting later.

In [ ]:
site_department_month_features = make_site_department_month_features(
    incident_enriched=incident_compact,
    task_enriched=task_compact,
    audit_enriched=audit_compact,
    active_only=ACTIVE_ONLY,
)

display_df("Site / department / month feature table", site_department_month_features, n=DISPLAY_ROWS)


## 14. Optional memory cleanup before EDA

In [ ]:
del incident_enriched, task_enriched, audit_enriched
del listitem_raw, location_raw, injury_raw, incident_raw, task_raw, audit_raw
gc.collect()

## 15. Generate EDA tables, plots, and markdown summary

The EDA function writes CSV tables, PNG plots, and a markdown summary into the output folder.

In [ ]:
create_eda_outputs(
    raw_shapes=raw_shapes,
    incident_enriched=incident_compact,
    pattern_records=pattern_records,
    injury_agg=injury_agg,
    task_enriched=task_compact,
    audit_enriched=audit_compact,
    site_month_features=site_department_month_features,
    output_dir=EDA_DIR,
)

print(f"EDA outputs written to: {EDA_DIR}")

## 16. Display generated EDA summary

In [ ]:
summary_path = EDA_DIR / "eda_summary.md"
if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding="utf-8")))
else:
    print("EDA summary file was not created.")

## 17. Display generated EDA tables

In [ ]:
tables_dir = EDA_DIR / "tables"
for table_path in sorted(tables_dir.glob("*.csv")):
    show_csv_table(table_path, n=DISPLAY_ROWS)

## 18. Display generated EDA plots

In [ ]:
plots_dir = EDA_DIR / "plots"
for plot_path in sorted(plots_dir.glob("*.png")):
    show_plot(plot_path)

## 19. Final output files created by this notebook

In [ ]:
display(Markdown("### Processed output files"))
for path in sorted(PROCESSED_DIR.glob("*")):
    print(path)

print("\nEDA output files:")
for path in sorted(EDA_DIR.rglob("*")):
    if path.is_file():
        print(path)

## 20. Modeling handoff: what to use next

For the first unsupervised pattern-learning model, use `outputs/processed/pattern_learning_records.csv`.

Recommended first model input: `ml_text_early`

Recommended first baseline: `TF-IDF + MiniBatchKMeans`

Recommended stronger model: `Sentence embeddings + HDBSCAN or BERTopic`

Expected model outputs: cluster ID, cluster label, representative examples, top keywords / hazard tags, site concentration, department concentration, monthly trend, and similar historical records.